**Figure 1: Overhead Accumulation and Optimization.** The red curve walks the
optimization chain of full-mode AgentTX on the deterministic 64-call workload
(2 repeats); horizontal lines mark the current baselines (`shared_try` is within 3%
of `per-call try` and is omitted). Results suggest that repeated per-call userspace
setup, not isolation itself, accounts for the main overhead: the persistent try
worker alone cuts latency by ~61%, ending 3.0x over bare, while the checkpoint
floor shows isolation itself costs only ~1.3x.


In [ ]:
# ipython -c "%run plot.ipynb"
# FAST/USENIX line-plot conventions: white panels, boxed top legend, red solid squares = ours.
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

STANDARD_WIDTH = 17.8            # USENIX two-column text width, cm

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

OURS  = dict(color='#c00000', marker='s', linestyle='-',  linewidth=1.0, markersize=3.2)
BASE1 = dict(color='#e78129', marker='x', linestyle=':',  linewidth=0.9, markersize=3.6, markeredgewidth=0.9)
BASE2 = dict(color='#4f9fcf', marker='^', linestyle='-.', linewidth=0.9, markersize=3.2, markerfacecolor='none')
REF   = dict(color='black', linestyle='--', linewidth=0.8)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

history = pd.read_csv(RESULTS / 'motivation_optimization_history.csv')
runtime = pd.read_csv(RESULTS / 'motivation_runtime_comparison.csv').set_index('mode')
chain = history[history['metric'] == 'full_ms_per_step']
y = [float(chain['before'].iloc[0])] + [float(v) for v in chain['after']]
labels = ['naive', 'W/D\ntrace', 'READ/NEG\neffects', 'script\nreuse', 'defer\nGC', 'direct\nscript', 'persistent\nworker']

bare = float(runtime.loc['bare', 'per_step_mean_ms'])
floor = float(runtime.loc['shared_checkpoint', 'per_step_mean_ms'])
percall = float(runtime.loc['per_call_try', 'per_step_mean_ms'])

fig, ax = plt.subplots(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH / 2), cm_to_inch(4.6)))
x = np.arange(len(y))
line_ours, = ax.plot(x, y, **OURS, label='AgentTX full (ours)', zorder=3)
def hline_style(base):
    return {k: v for k, v in base.items() if not k.startswith('marker')}

line_percall = ax.axhline(percall, **hline_style(BASE1), label='per-call try')
line_floor = ax.axhline(floor, **hline_style(BASE2), label='checkpoint floor')
line_bare = ax.axhline(bare, **REF, label='bare')
ax.annotate(f"{y[-1]:.0f} ms ({y[-1] / bare:.1f}x bare)", (x[-1], y[-1]),
            textcoords='offset points', xytext=(2, -11), ha='right', fontsize=6, color=OURS['color'])
ax.set_xticks(x, labels=labels, fontsize=6)
ax.set_ylabel('Latency (ms/step)', fontsize=8)
ax.set_ylim(0, max(y) * 1.15)
ax.tick_params(axis='y', labelsize=7)
ax.legend(handles=[line_ours, line_percall, line_floor, line_bare], loc='upper center',
          bbox_to_anchor=(0.5, 1.26), ncol=4, fontsize=6, columnspacing=0.8,
          handlelength=1.6, handletextpad=0.4, borderpad=0.3)
plt.tight_layout(pad=0.3)
plt.savefig(FIGDIR / 'FIG-Motivation-Optimization.pdf', bbox_inches='tight', pad_inches=0.02)
plt.savefig(FIGDIR / 'FIG-Motivation-Optimization.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

print(f"chain: {y[0]:.1f} -> {y[-1]:.1f} ms/step ({(1 - y[-1] / y[0]):.1%}); full = {y[-1] / bare:.2f}x bare, floor = {floor / bare:.2f}x bare")
